In [ ]:
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics
import pandas as pd
import glob
import xml.etree.ElementTree as ET
import nltk
import math
import numpy as np
import warnings
from sklearn.model_selection import cross_validate


warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline, Pipeline
from importlib import reload

In [ ]:
import dodflib.core.loader as LD
from dodflib.core.utils import pad_dataframe

import dodflib.core.preprocess as P
import dodflib.core.feature_extraction as FT
import dodflib.core.utils as UT
import dodflib.core.transformers as T

from importlib import reload as rl

In [ ]:
anc = 'Ato_Nomeacao_Comissionado'
aec = 'Ato_Exoneracao_Comissionado'

In [ ]:
reload(LD)
rl(UT)
p = ['xmls/10_193.17.9.2013_v1.xml']
aec = 'Ato_Exoneracao_Comissionado' 
ldr = LD.LoaderXML(p)

In [ ]:
# ldr.all_columns

In [ ]:
reload(P)
reload(UT)
reload(FT)

params = dict(
    algorithm = 'l2sgd', 
    c2=1,
    max_iterations=10, 
    all_possible_transitions=True,
)

model = sklearn_crfsuite.CRF(
    **params,
    verbose=False
)


to_drop=r'(annotator|offset)'
pipe_crf = Pipeline([
    ('drop_pattern', T.DropPattern(to_drop)),
    ('selector', T.SelectColumn(aec)),
    ('tokenizer', T.Tokenizer()),
    ('crformer', T.CRFFeaturizer()),
    ('model', sklearn_crfsuite.CRF(**params, verbose=False)),
])

pipe_labelizer = Pipeline([
    ('drop_pattern', T.DropPattern(to_drop)),
    ('keep_only', T.KeepOnlyOneAct(aec)),
    ('iobifier', T.IOBifyer(column=aec)),
])

df = ldr[aec]
train_y=pipe_labelizer.fit_transform(df);
pipe_crf.fit(df, train_y);

In [ ]:
import math
training_ratio = 0.7
limit = int(math.ceil(training_ratio * df.shape[0]))


df_train, df_test = df.iloc[:limit], df.iloc[limit:]
y = pipe_labelizer.fit_transform(df_test)

pipe_crf.fit(
    df_train,
    pipe_labelizer.fit_transform(df_train)
);
y_pred=pipe_crf.predict(df_test);

In [ ]:
y